In [ ]:
import trimesh
import numpy as np
from neural_poisson.data.binovox import read_as_3d_array
from pathlib import Path
import matplotlib.pyplot as plt
import open3d as o3d

# define the paths
model_id = "107bce22d72f322eedf1bb0b62653056"
root_dir = Path("/home/borth/2d-gaussian-splatting/")
shapenet_dir = root_dir / "data/ShapeNetCore/04256520" /  model_id


def sample_points_inside_surface(shapenet_path: str, max_samples: int = 100_000):
    shapenet_dir = Path(shapenet_path)

    # load the mesh
    mesh_path = shapenet_dir / "models/model_normalized.obj"
    mesh = o3d.io.read_triangle_mesh(str(mesh_path))

    # load the surface voxelization
    surface_path = shapenet_dir / "models/model_normalized.surface.binvox"
    with open(str(surface_path), 'rb') as f:
        surface = read_as_3d_array(f)

    # load the solid voxelization
    solid_path = shapenet_dir / "models/model_normalized.solid.binvox"
    with open(str(solid_path), 'rb') as f:
        solid = read_as_3d_array(f)

    # compute the inside voxels
    inside = (~surface.data) & solid.data
    voxel_size = inside.shape[0]

    # compute the axis
    i_n = ((np.linspace(0, voxel_size, voxel_size)) + 0.5) / voxel_size
    xs = surface.scale  * i_n + surface.translate[0]
    ys = surface.scale  * i_n + surface.translate[1]
    zs = surface.scale  * i_n + surface.translate[2]

    # compute the grid in world space of the mesh
    xs, ys, zs = np.meshgrid(xs, ys, zs, indexing="ij")
    grid = np.stack((xs.ravel(), ys.ravel(), zs.ravel()), axis=-1)
    grid = grid.reshape(voxel_size, voxel_size, voxel_size, 3)

    # compute the points on the grid
    inside_points = grid[inside]
    idx = np.random.permutation(len(inside_points))[:max_samples]
    return inside_points[idx]
    
# axis_surface = surface.data[10, :, :]
# plt.imshow(axis_surface)
# plt.show()

# axis_solid = solid.data[10, :, :]
# plt.imshow(axis_solid)
# plt.show()

inside_points = sample_points_inside_surface(shapenet_dir, 100_000)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(inside_points)
o3d.visualization.draw_plotly([pcd])

In [ ]:
voxel_size = inside.shape[0]

# extract the boundaries from the mesh
vertices = np.asarray(mesh.vertices)
x_min = vertices[:, 0].min()
x_max = vertices[:, 0].max()
y_min = vertices[:, 1].min()
y_max = vertices[:, 1].max()
z_min = vertices[:, 2].min()
z_max = vertices[:, 2].max()

# compute the axis
xs = np.linspace(x_min, x_max, voxel_size)
ys = np.linspace(y_min, y_max, voxel_size)
zs = np.linspace(z_min, z_max, voxel_size)

# compute the grid in world space of the mesh
xs, ys, zs = np.meshgrid(xs, ys, zs, indexing="ij")
grid = np.stack((xs.ravel(), ys.ravel(), zs.ravel()), axis=-1)
grid = grid.reshape(voxel_size, voxel_size, voxel_size, 3)

inside_points = grid[surface.data]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(inside_points)
o3d.visualization.draw_plotly([pcd, mesh])

In [ ]:
voxel_size = inside.shape[0]

# compute the axis
scale = surface.scale 
translate = surface.translate
i_n = ((np.linspace(0, voxel_size, voxel_size)) + 0.5) / voxel_size

xs = scale * i_n + translate[0]
ys = scale * i_n + translate[1]
zs = scale * i_n + translate[2]

# # compute the grid in world space of the mesh
xs, ys, zs = np.meshgrid(xs, ys, zs, indexing="ij")
grid = np.stack((xs.ravel(), ys.ravel(), zs.ravel()), axis=-1)
grid = grid.reshape(voxel_size, voxel_size, voxel_size, 3)

inside_points = grid[inside]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(inside_points)
o3d.visualization.draw_plotly([pcd, mesh])

In [ ]:
inside_points

In [ ]:
o3d.visualization.draw_plotly([pcd])

In [ ]:
y_min

In [ ]:
surface.scale

In [ ]:
np.array(surface.translate) * 2

In [ ]:
solid.translate

In [ ]:
mesh = o3d.io.read_triangle_mesh(str(shapenet_path))
voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh(mesh, 0.01)
voxel_grid.create_dense()

In [ ]:
import trimesh
import numpy as np
import open3d as o3d

import trimesh
import numpy as np
from neural_poisson.data.binovox import read_as_3d_array
from pathlib import Path
import matplotlib.pyplot as plt
import open3d as o3d

# define the paths
model_id = "107bce22d72f322eedf1bb0b62653056"
root_dir = Path("/home/borth/2d-gaussian-splatting/")
shapenet_dir = root_dir / "data/ShapeNetCore/04256520"
shapenet_path = shapenet_dir / model_id / "models/model_normalized.obj"


# Load mesh
mesh = trimesh.load_mesh(str(shapenet_path))

# Define voxel grid resolution
voxel_size = 0.01  # Adjust based on your needs
bounds_min, bounds_max = mesh.bounds
grid_x, grid_y, grid_z = np.mgrid[
    bounds_min[0]:bounds_max[0]:voxel_size, 
    bounds_min[1]:bounds_max[1]:voxel_size, 
    bounds_min[2]:bounds_max[2]:voxel_size
]

# Stack grid into (N,3) coordinates
voxel_points = np.vstack((grid_x.ravel(), grid_y.ravel(), grid_z.ravel())).T

# Compute signed distance values
sdf_values = trimesh.proximity.signed_distance(mesh, voxel_points)

In [ ]:
surface_path = shapenet_dir / model_id / "models/model_normalized.surface.binvox"
with open(str(surface_path), 'rb') as f:
    surface = read_as_3d_array(f)


In [ ]:
from neural_poisson.data.binovox import read_as_3d_array


from pathlib import Path
import matplotlib.pyplot as plt

model_id = "107bce22d72f322eedf1bb0b62653056"
root_dir = Path("/home/borth/2d-gaussian-splatting/")
shapenet_dir = root_dir / "data/ShapeNetCore/04256520"
shapenet_path = shapenet_dir / model_id / "models/model_normalized.surface.binvox"
shapenet_path = shapenet_dir / model_id / "models/model_normalized.solid.binvox"
with open(str(shapenet_path), 'rb') as f:
    m = read_as_3d_array(f)

axis = m.data[:, 110, :]
plt.imshow(axis)

In [ ]:
from neural_poisson.data.grid import coord_grid, coord_grid_along_axis

grid = coord_grid_along_axis(
    axis="x",
    voxel_size=128,
    domain=(-1.0, 1.0),
    default_coord=0.0,
    device="cpu",
)
